# Gauge Reader — train the models on Colab

Trains the three models (docs/PLAN.md §4) on a Colab GPU and evaluates the full system on synth-test (E2):

| Model | Script | Rough time on a T4 |
|---|---|---|
| A — dial detector + segmenter (YOLO-seg) | `models.train_seg` | 2–3 h |
| B — 7 keypoints (YOLO-pose) | `models.train_pose` | 1.5–3 h |
| C — direct regression, S3 (ResNet-18) | `models.train_regress` | 0.5–1 h |

**Total wall time on a T4 ≈ 5–7.5 h** (estimates; training times not yet measured):
setup ~3 min · data ~5 min from `cv-dial-data.zip` (first-time generation on Colab instead: +40–55 min) ·
smoke test ~2 min · training 4–7 h · E2 evaluation + plots ~5–10 min.
Free Colab sessions rarely last that long: plan **session 1** = data + seg, **session 2** = data + pose +
regression + E2. Finished runs are on Drive, and step 8 finds them in any later session.

This notebook only *calls the repo's scripts* (CLAUDE.md: no notebook-only logic). Everything it writes under
`results/` goes to Google Drive (`MyDrive/cv-dial/results/`), so a disconnect never loses finished runs.

**Runtime → Change runtime type → GPU (T4 or better)** before running anything.

### Before you start — on your laptop, in the repo

```bash
git archive -o cv-dial-src.zip HEAD                      # the code (or push to GitHub and use CODE_SOURCE="git")

uv run python -m gen.generate --config configs/gen/default.yaml --split train --n 20000 --out data/synth/train
uv run python -m gen.generate --config configs/gen/default.yaml --split val   --n 2000  --out data/synth/val
uv run python -m gen.export --src data/synth/train
uv run python -m gen.export --src data/synth/val
zip -r -0 cv-dial-data.zip data/synth_seg data/synth_crops data/synth/test   # ~2.4 GB, -0 = store (JPEGs)
```

Upload `cv-dial-src.zip` and `cv-dial-data.zip` to a Google Drive folder **`MyDrive/cv-dial/`**.
(`data/synth/test` is the stratified synth-test used by the E2 evaluation at the end.)

Generating on a laptop is the fast route: ~10 min for all 24k images on a 10-core Mac, versus ~30–40 min on
Colab's 2 CPUs (while the GPU sits idle). If you do set `DATA_SOURCE = "generate"`, the notebook generates
**once** and saves `cv-dial-data.zip` to Drive, so every later session just unzips it.

## 1. Settings

In [ ]:
DRIVE_DIR   = "/content/drive/MyDrive/cv-dial"   # holds the zips; results/ is created here
CODE_SOURCE = "drive_zip"   # "drive_zip": DRIVE_DIR/cv-dial-src.zip   |  "git": clone GIT_URL
GIT_URL     = ""            # e.g. "https://github.com/<you>/cv-dial.git" (private repos need a token)
DATA_SOURCE = "drive_zip"   # "drive_zip": DRIVE_DIR/cv-dial-data.zip  |  "generate": build it here once, then cached
N_TRAIN, N_VAL, N_TEST = 20000, 2000, 2000      # only used when DATA_SOURCE == "generate"
USE_DTD     = True          # generate only: DTD texture backgrounds (archive cached on Drive after one download)
WORKERS     = 2             # Colab has 2 CPU cores; more data-loader workers just thrash
REPO        = "/content/cv-dial"

## 2. Google Drive and GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"

## 3. Code and dependencies

In [ ]:
import os, shutil, subprocess
if os.path.exists(REPO):
    shutil.rmtree(REPO)
if CODE_SOURCE == "git":
    subprocess.run(["git", "clone", "--depth", "1", GIT_URL, REPO], check=True)
else:
    os.makedirs(REPO)
    subprocess.run(["unzip", "-q", f"{DRIVE_DIR}/cv-dial-src.zip", "-d", REPO], check=True)
%cd {REPO}
# Colab already ships torch, torchvision, OpenCV, NumPy, matplotlib, Pillow and PyYAML; only Ultralytics is missing.
# (Reinstalling torch from requirements.txt would download ~2 GB and risk a CUDA mismatch.)
!pip install -q "ultralytics==8.4.178"
!python -c "import ultralytics, cv2, numpy; print('ultralytics', ultralytics.__version__, '| opencv', cv2.__version__, '| numpy', numpy.__version__)"
!git log -1 --oneline 2>/dev/null || echo "(code from zip)"

## 4. Results → Google Drive

`results/` becomes a link to `MyDrive/cv-dial/results/`, so weights, metrics and plots persist across sessions.

In [ ]:
!rm -rf results && ln -s {DRIVE_DIR}/results results && ls -la results

## 5. Data

* An existing `DRIVE_DIR/cv-dial-data.zip` is always used (unzipped straight from Drive to the local disk —
  training directly from Drive would be very slow). Takes a few minutes.
* Otherwise, with `DATA_SOURCE = "generate"`, the data is generated here **once** (~30–40 min on 2 CPUs) and
  zipped back to Drive, so the next session skips generation.

In [ ]:
import os, subprocess, time
t0 = time.time()
DATA_ZIP = f"{DRIVE_DIR}/cv-dial-data.zip"
if os.path.exists(DATA_ZIP):
    print("unzipping", DATA_ZIP)
    subprocess.run(["unzip", "-q", "-o", DATA_ZIP, "-d", REPO], check=True)      # no intermediate copy
elif DATA_SOURCE == "generate":
    if USE_DTD and not os.path.isdir("data/backgrounds/dtd"):
        DTD_TAR = f"{DRIVE_DIR}/dtd-r1.0.1.tar.gz"
        if not os.path.exists(DTD_TAR):                                          # download once, keep on Drive
            !curl -fL --retry 5 -C - -o /content/dtd.tar.gz https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz
            !cp /content/dtd.tar.gz "{DTD_TAR}"
        else:
            !cp "{DTD_TAR}" /content/dtd.tar.gz
        !mkdir -p /content/dtd && tar -xzf /content/dtd.tar.gz -C /content/dtd --wildcards 'dtd/images/*.jpg'
        !mkdir -p data/backgrounds/dtd && cp -R /content/dtd/dtd/images/. data/backgrounds/dtd/
    !python -m gen.generate --config configs/gen/default.yaml --split train --n {N_TRAIN} --out data/synth/train
    !python -m gen.generate --config configs/gen/default.yaml --split val --n {N_VAL} --out data/synth/val
    !python -m gen.generate --config configs/gen/default.yaml --split test --n {N_TEST} --out data/synth/test --stratify-obliquity
    !python -m gen.export --src data/synth/train
    !python -m gen.export --src data/synth/val
    print("saving the data to Drive for later sessions ...")
    !zip -q -r -0 /content/cv-dial-data.zip data/synth_seg data/synth_crops data/synth/test && cp /content/cv-dial-data.zip "{DATA_ZIP}" && rm /content/cv-dial-data.zip
else:
    raise FileNotFoundError(f"{DATA_ZIP} not found: upload it, or set DATA_SOURCE = 'generate'")
for d in ["data/synth_seg/images/train", "data/synth_seg/images/val", "data/synth_crops/images/train",
          "data/synth_crops/images/val", "data/synth/test/records"]:
    print(f"{d:35s} {len(os.listdir(d)) if os.path.isdir(d) else 'MISSING'}")
print(f"data ready in {(time.time() - t0) / 60:.1f} min")

## 6. Smoke test (~1 min)

Catches environment problems before hours of training. Uses a tiny self-generated dataset; its numbers mean nothing.

In [ ]:
!python -m models.train_pose --config configs/train/pose.yaml --smoke --set train.device=0 2>&1 | tail -3

## 7. Train

Run the three cells in any order, in one session or across several. Each run gets its own folder
`results/<name>_<timestamp>/` with the resolved `config.yaml`, `run.json` (status, weights path) and the weights.

`fliplr: 0.0` is enforced for pose and regression (CLAUDE.md invariant 2); the scripts refuse to start otherwise.
Extra overrides go after `--set`, e.g. `train.epochs=30 train.batch=32`.

In [ ]:
# Model A: detector + segmenter (~2-3 h on a T4)
!python -m models.train_seg --config configs/train/seg.yaml --set train.workers={WORKERS}

In [ ]:
# Model B: keypoints (~1.5-3 h; early stopping after 30 epochs without improvement)
!python -m models.train_pose --config configs/train/pose.yaml --set train.workers={WORKERS}

In [ ]:
# Model C: direct regression for S3 (~0.5-1 h)
!python -m models.train_regress --config configs/train/regress.yaml --set train.workers={WORKERS}

### If the session dies mid-run

* **Seg / pose:** Ultralytics can resume from the last epoch:
  `!yolo train resume model=results/<run>/ultralytics/weights/last.pt`
* **Regression:** rerun the cell (it is the short one).

Finished runs are already on Drive.

## 8. Latest trained weights

In [ ]:
import glob, json, os
def latest(name):
    """Newest finished, non-smoke run of `name` that has weights."""
    runs = []
    for p in glob.glob(f"results/{name}_*/run.json"):
        info = json.load(open(p))
        if "_smoke_" not in p and info.get("status") == "done" and os.path.exists(info.get("weights", "")):
            runs.append((os.path.getmtime(p), info["weights"]))
    return max(runs)[1] if runs else None

W = {k: latest(k) for k in ("seg", "pose", "regress")}
for k, v in W.items():
    print(f"{k:8s} {v or 'not trained yet'}")

## 9. E2 — full system on synth-test

Every reported number comes from `eval.evaluate` (CLAUDE.md invariant 7). Writes `predictions.csv`,
`metrics.csv`, `keypoints.csv` and plots into `results/e2_<timestamp>/`.

In [ ]:
import time
assert all(W.values()), f"train all three models first: {W}"
RUN = f"results/e2_{time.strftime('%Y%m%d-%H%M%S')}"
SEG_W, POSE_W, REG_W = W["seg"], W["pose"], W["regress"]
!python -m eval.evaluate --run {RUN} --split synth-test --seg-weights {SEG_W} --pose-weights {POSE_W} --regress-weights {REG_W}
!python -m eval.plots run --run {RUN}

In [ ]:
from IPython.display import Image, display
display(Image(f"{RUN}/plots/mae_vs_obliquity.png"))
display(Image(f"{RUN}/plots/mae_bins.png"))

## 10. Use the weights locally

Everything is in `MyDrive/cv-dial/results/`. Download the three run folders into your repo's `results/`, then:

```bash
uv run python -m pipeline.demo_app \
    --seg-weights results/<seg_run>/ultralytics/weights/best.pt \
    --pose-weights results/<pose_run>/ultralytics/weights/best.pt \
    --regress-weights results/<regress_run>/weights/best.pt
```

The paths are printed in step 8. The real-photo evaluation (E3) runs the same way with `--split real`
once `data/real/` exists — locally is fine, it needs no GPU.